In [1]:
# ============================================================
# PHASE 1 — Nettoyage + Feature Engineering
# Tunis (2010–2024)
# ============================================================

# --- Imports ---
import numpy as np
import pandas as pd
from pathlib import Path

# --- Paramètres d'affichage (facilite la lecture dans Jupyter) ---
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# --- Chemins des fichiers ---
SRC_PATH   = "POWER_Point_Daily_20100101_20241231_036d81N_010d18E_LST.csv"  # Fichier source NASA POWER
CLEAN_PATH = "tunis_2010_2024_clean.csv"                                   # Export nettoyé
FEAT_PATH  = "tunis_2010_2024_clean_features.csv"                          # Export nettoyé + features

# --- Seuils (à justifier dans le rapport) ---
RAIN_MM   = 1.0   # mm/jour : au-dessus => "Pluvieux"
SUN_MJ    = 18.0  # seuil rayonnement (unité NASA POWER) => "Ensoleillee"
HEAT_C    = 35.0  # °C : vague de chaleur si T2M_MAX >= 35
FREEZE_C  = 0.0   # °C : gel si T2M_MIN < 0

# --- Petit logger pour rendre l'exécution claire ---
def log_ok(msg):   print(f"✅ {msg}")
def log_warn(msg): print(f"⚠️ {msg}")
def log_err(msg):  print(f"❌ {msg}")


# ============================================================
# Utils (gestion d’erreurs + helpers)
# ============================================================

def assert_file_exists(path: str):
    # Vérifie que le fichier existe (sinon erreur claire)
    if not Path(path).exists():
        raise FileNotFoundError(f"Fichier introuvable: {path}")

def export_csv(df: pd.DataFrame, path: str):
    # Export CSV en utf-8-sig (bien pour Excel et accents)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    log_ok(f"Export -> {path}")

def require_columns(df: pd.DataFrame, cols: list, context: str):
    # Vérifie la présence de colonnes indispensables
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{context} : colonnes manquantes -> {missing}")

def to_numeric_cols(df: pd.DataFrame, cols: list):
    # Convertit les colonnes en numériques (si texte => NaN)
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


In [2]:
# ============================================================
# Lecture NASA POWER (fichier Daily Point) + reconstruction date
# ============================================================

def read_power_csv_with_header(path: str) -> pd.DataFrame:
    # Certains fichiers NASA POWER contiennent un header texte "-BEGIN HEADER-"..."-END HEADER-"
    # On détecte "-END HEADER-" puis on lit le CSV juste après.
    assert_file_exists(path)

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    end_header_idx = None
    for i, line in enumerate(lines):
        if line.strip() == "-END HEADER-":
            end_header_idx = i
            break

    # Si on ne trouve pas -END HEADER-, on tente une lecture CSV directe
    if end_header_idx is None:
        log_warn("Header NASA POWER non détecté, lecture CSV directe.")
        return pd.read_csv(path)

    # Lecture du fichier en sautant le header texte
    return pd.read_csv(path, skiprows=end_header_idx + 1)

def build_date_column(df: pd.DataFrame) -> pd.DataFrame:
    # NASA POWER fournit souvent YEAR + DOY (day of year)
    # On reconstruit une vraie date (datetime)
    if ("YEAR" in df.columns) and ("DOY" in df.columns):
        df["date"] = pd.to_datetime(
            df["YEAR"].astype(int).astype(str) + df["DOY"].astype(int).astype(str).str.zfill(3),
            format="%Y%j",
            errors="coerce"
        )
    else:
        # Fallback si jamais YEAR/DOY n'existent pas (rare)
        date_col = None
        for c in ["YYYYMMDD", "DATE", "date"]:
            if c in df.columns:
                date_col = c
                break

        if date_col is None:
            raise ValueError("Pas de YEAR/DOY ni colonne date (YYYYMMDD/DATE/date).")

        # Conversion selon le format de la colonne trouvée
        if date_col == "YYYYMMDD":
            df["date"] = pd.to_datetime(df[date_col].astype(str), format="%Y%m%d", errors="coerce")
        else:
            df["date"] = pd.to_datetime(df[date_col], errors="coerce")

    # On garde uniquement les lignes avec date valide, et on trie
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df



In [3]:
# ============================================================
# Nettoyage / préparation
# ============================================================

def clean_meteo_data(df: pd.DataFrame) -> pd.DataFrame:
    # Vérifie que la colonne date existe
    require_columns(df, ["date"], "clean_meteo_data")

    # Liste des colonnes météo attendues (selon NASA POWER)
    meteo_cols = [c for c in ["T2M_MAX","T2M_MIN","T2M","RH2M","WS2M","PRECTOTCORR","ALLSKY_SFC_SW_DWN"] if c in df.columns]
    if not meteo_cols:
        raise ValueError("Aucune colonne météo reconnue.")

    # Conversion en numérique
    df = to_numeric_cols(df, meteo_cols)

    # Remplacement des valeurs sentinelles NASA POWER (-999, etc.) par NaN
    df[meteo_cols] = df[meteo_cols].replace([-999, -999.0, -9999, -9999.0], np.nan)

    # Règles de cohérence / bornes plausibles
    if "RH2M" in df.columns:
        # Humidité relative doit être entre 0 et 100
        df.loc[(df["RH2M"] < 0) | (df["RH2M"] > 100), "RH2M"] = np.nan

    if "PRECTOTCORR" in df.columns:
        # Pluie négative impossible
        df.loc[df["PRECTOTCORR"] < 0, "PRECTOTCORR"] = np.nan

    if "WS2M" in df.columns:
        # Vent négatif impossible
        df.loc[df["WS2M"] < 0, "WS2M"] = np.nan

    if "ALLSKY_SFC_SW_DWN" in df.columns:
        # Rayonnement négatif impossible
        df.loc[df["ALLSKY_SFC_SW_DWN"] < 0, "ALLSKY_SFC_SW_DWN"] = np.nan

    if ("T2M_MIN" in df.columns) and ("T2M_MAX" in df.columns):
        # T_min ne doit pas dépasser T_max
        bad = df["T2M_MIN"] > df["T2M_MAX"]
        df.loc[bad, ["T2M_MIN","T2M_MAX"]] = np.nan

    # --- Compléter le calendrier journalier ---
    df = df.set_index("date")
    df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="D"))
    df.index.name = "date"

    # --- Remplissage des valeurs manquantes ---
    # 1) interpolation temporelle
    # 2) fallback : forward fill puis backward fill
    for c in meteo_cols:
        df[c] = df[c].interpolate(method="time")
        df[c] = df[c].ffill().bfill()

    return df.reset_index()



In [4]:
# ============================================================
# Feature Engineering
# ============================================================

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    # Vérifie la présence de la date
    require_columns(df, ["date"], "add_features")

    # --- Variables temporelles ---
    df["annee"] = df["date"].dt.year
    df["mois"] = df["date"].dt.month
    df["jour"] = df["date"].dt.day
    df["jour_semaine"] = df["date"].dt.dayofweek  # 0=lundi ... 6=dimanche

    # --- Saison (sans accent pour éviter problèmes d'encodage) ---
    def saison_from_month(m):
        if m in [12, 1, 2]: return "Hiver"
        if m in [3, 4, 5]: return "Printemps"
        if m in [6, 7, 8]: return "Ete"
        return "Automne"

    df["saison"] = df["mois"].apply(saison_from_month)

    # --- Type_de_jour basé sur pluie + rayonnement ---
    df["Type_de_jour"] = "Nuageuse"  # valeur par défaut

    # Si pluie >= 1 mm => pluvieux
    if "PRECTOTCORR" in df.columns:
        df.loc[df["PRECTOTCORR"] >= RAIN_MM, "Type_de_jour"] = "Pluvieux"

    # Sinon si rayonnement élevé => ensoleillé
    if "ALLSKY_SFC_SW_DWN" in df.columns:
        not_rain = True
        if "PRECTOTCORR" in df.columns:
            not_rain = (df["PRECTOTCORR"] < RAIN_MM)

        df.loc[not_rain & (df["ALLSKY_SFC_SW_DWN"] >= SUN_MJ), "Type_de_jour"] = "Ensoleillee"

    # --- Indicateurs climatiques ---
    df["Vague_de_chaleur"] = (df["T2M_MAX"] >= HEAT_C).astype(int) if "T2M_MAX" in df.columns else 0
    df["Jour_de_gel"] = (df["T2M_MIN"] < FREEZE_C).astype(int) if "T2M_MIN" in df.columns else 0

    # --- Features ML-safe (uniquement passé) ---
    # Lag 1 (valeur de la veille) + moyenne glissante 7 jours
    if "T2M_MAX" in df.columns:
        df["T2M_MAX_lag1"] = df["T2M_MAX"].shift(1).bfill()
        df["T2M_MAX_roll7"] = df["T2M_MAX"].rolling(7, min_periods=1).mean()

    if "PRECTOTCORR" in df.columns:
        df["PRECTOTCORR_roll7"] = df["PRECTOTCORR"].rolling(7, min_periods=1).mean()

    return df


In [5]:
# ============================================================
# RUN PHASE 1
# ============================================================

def run_phase1():
    try:
        # 1) Lire le fichier source
        log_ok("Lecture fichier source NASA POWER")
        df = read_power_csv_with_header(SRC_PATH)

        # 2) Construire la colonne date
        df = build_date_column(df)
        log_ok(f"Période détectée: {df['date'].min()} -> {df['date'].max()} | shape={df.shape}")

        # 3) Nettoyer + préparer
        log_ok("Nettoyage / préparation")
        df_clean = clean_meteo_data(df)

        # 4) Export des données nettoyées
        export_csv(df_clean, CLEAN_PATH)

        # 5) Feature engineering
        log_ok("Feature engineering")
        df_feat = add_features(df_clean)

        # 6) Export des données nettoyées + features
        export_csv(df_feat, FEAT_PATH)

        # 7) Retourner le dataset final phase 1
        log_ok("Phase 1 terminée")
        return df_feat

    except Exception as e:
        # Gestion d’erreur claire
        log_err(f"Phase 1 échouée: {e}")
        raise
# --- Exécution ---
df_phase1 = run_phase1()

# --- Aperçu ---
df_phase1.head()


✅ Lecture fichier source NASA POWER
⚠️ Header NASA POWER non détecté, lecture CSV directe.
❌ Phase 1 échouée: Pas de YEAR/DOY ni colonne date (YYYYMMDD/DATE/date).


ValueError: Pas de YEAR/DOY ni colonne date (YYYYMMDD/DATE/date).